In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),

            nn.Linear(512, 512),
            nn.ReLU(),

            nn.Linear(512, 10),   
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [2]:
train_dl = DataLoader(datasets.FashionMNIST(
    root = "data",
    train = True,
    transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale = True)
    ])
), shuffle = True, batch_size = 64)

In [3]:
test_dl = DataLoader(datasets.FashionMNIST(
    root = "data",
    train = False,
    transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale = True)
    ])
), shuffle = True)

In [18]:
def train_loop(dataloader, model: nn.Module, loss_fn, optimize, device):
    model.train()
    size = len(dataloader.dataset)
    for batch, (x, y) in enumerate(dataloader, start = 1):
        x, y = x.to(device), y.to(device)
        # forward
        pred = model(x)
        loss = loss_fn(pred, y)

        # backward
        loss.backward()

        # gradient descent
        optimizer.step()

        # clear gradients
        optimizer.zero_grad()

        if batch % 100 == 0: # show stats every 100 batches
            loss, current = loss.item(), min(batch * dataloader.batch_size, size)
            print(f"iteration {current} / {size}, avg loss: {loss}")

In [19]:
def test_loop(dataloader, model, loss_fn, device):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            test_loss += loss_fn(pred, y)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(f"Test stats: \nAccuracy: {correct * 100}; avg loss: {test_loss}\n")

In [7]:
model = NeuralNetwork()

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

for epoch in range(15):
    print(f"Epoch {epoch + 1}", "-" * 50, sep = "")
    train_loop(train_dl, model, loss_fn, optimizer)
    test_loop(test_dl, model, loss_fn)
print("Done!")

Epoch 1--------------------------------------------------
iteration 6400 / 60000, avg loss: 2.2676236629486084
iteration 12800 / 60000, avg loss: 2.2735707759857178
iteration 19200 / 60000, avg loss: 2.241086483001709
iteration 25600 / 60000, avg loss: 2.2457306385040283
iteration 32000 / 60000, avg loss: 2.2318193912506104
iteration 38400 / 60000, avg loss: 2.217719078063965
iteration 44800 / 60000, avg loss: 2.1668782234191895
iteration 51200 / 60000, avg loss: 2.1595733165740967
iteration 57600 / 60000, avg loss: 2.132352352142334


KeyboardInterrupt: 

In [20]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_linear = nn.Sequential(
            nn.Conv2d(1, 16, 3), 
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(800, 128),
            nn.ReLU(),

            #nn.Linear(512, 128),
            #nn.ReLU(),

            nn.Linear(128, 10),
        )
    def forward(self, x):
        logits = self.conv_linear(x)
        return logits

In [21]:
model = NeuralNetwork()
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
model.to(device)

NeuralNetwork(
  (conv_linear): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=800, out_features=128, bias=True)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [25]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

for epoch in range(15):
    print(f"Epoch {epoch + 1}", "-" * 50, sep = "")
    train_loop(train_dl, model, loss_fn, optimizer, device)
    test_loop(test_dl, model, loss_fn, device)
print("Done!")

Epoch 1--------------------------------------------------
iteration 6400 / 60000, avg loss: 0.6344956755638123
iteration 12800 / 60000, avg loss: 0.5655512809753418
iteration 19200 / 60000, avg loss: 0.4518631100654602
iteration 25600 / 60000, avg loss: 0.4498187303543091
iteration 32000 / 60000, avg loss: 0.5016441345214844
iteration 38400 / 60000, avg loss: 0.5719711780548096
iteration 44800 / 60000, avg loss: 0.5271740555763245
iteration 51200 / 60000, avg loss: 0.4284611940383911
iteration 57600 / 60000, avg loss: 0.3644680380821228
Test stats: 
Accuracy: 85.14; avg loss: 0.4130396842956543

Epoch 2--------------------------------------------------
iteration 6400 / 60000, avg loss: 0.319925993680954
iteration 12800 / 60000, avg loss: 0.43343544006347656
iteration 19200 / 60000, avg loss: 0.36870360374450684
iteration 25600 / 60000, avg loss: 0.2875649631023407
iteration 32000 / 60000, avg loss: 0.29899656772613525
iteration 38400 / 60000, avg loss: 0.521216630935669
iteration 44800